# Matriz de priorización de estaciones para áreas verdes

Esta matriz no predice cuánto bajaría el PM10 en cada estación. Solo indica qué tan bien cumple cada estación las condiciones necesarias documentadas en la literatura.

Criterios calculados por estación:
1. **Régimen de humedad (RH)**: mediana y % de tiempo por encima del umbral de saturación (~75%) donde el beneficio de RH sobre PM10 reportado en la literatura desaparece.
2. **Régimen de ventilación (WSR)**: mediana y % de tiempo en condiciones de estancamiento (viento bajo), que favorecen acumulación de contaminantes sin importar la humedad.
3. **Fracción gruesa de PM10**: proporción de PM10 no explicada por PM2.5, como proxy de si domina partícula gruesa (más sensible al mecanismo de deposición) o fina (menos sensible).
4. **Concentración por dirección de viento**: qué tan dominada está la estación por una fuente de emisión direccional específica (menos favorable: sugiere que otra fuente domina sobre el efecto de vegetación).
5. **Cobertura y calidad de datos**: para saber qué tan confiable es cada estimación.

## Configuración

In [ ]:
STATIONS_FOLDER = "../data_estaciones_e4"
FILE_PATTERN = "*.csv"

# Umbral de RH (%) por encima del cual, según la literatura, el beneficio de humedad sobre PM10 deja de observarse.
RH_SATURATION_THRESHOLD = 75.0

# Percentil de WSR (calculado sobre el conjunto de TODAS las estaciones cargadas) que se usa para definir "condición de estancamiento".
WSR_STAGNATION_PERCENTILE = 10

# Pesos para el score de priorización.
PESOS = {
    "humedad": 0.25,      
    "ventilacion": 0.25,   
    "fraccion_gruesa": 0.25,  
    "fuente_direccional": 0.25,
}

assert abs(sum(PESOS.values()) - 1.0) < 1e-6, "Los pesos deben sumar 1"


## 1. Carga de datos

In [3]:
import pandas as pd
import numpy as np
import glob
import os
import re

pd.set_option('display.float_format', lambda x: f'{x:.3f}')


def extraer_nombre_estacion(filepath):
    """Heurística para obtener el código de estación a partir del nombre del archivo,
    ej. 'BD_NE2.csv' -> 'NE2'. Si el patrón no aplica, usa el nombre de archivo completo."""
    base = os.path.splitext(os.path.basename(filepath))[0]
    match = re.search(r'([A-Za-z]+\d+)$', base)
    return match.group(1) if match else base


def cargar_estacion(filepath):
    df = pd.read_csv(filepath)
    df.columns = [c.strip() for c in df.columns]
    df['date'] = pd.to_datetime(df['date'])
    df['estacion'] = extraer_nombre_estacion(filepath)
    return df


if not STATIONS_FOLDER:
    raise ValueError(
        "Falta especificar STATIONS_FOLDER en la celda de configuración de arriba "
        "(nombre o ruta de la carpeta con los CSV de las estaciones)."
    )

archivos = sorted(glob.glob(os.path.join(STATIONS_FOLDER, FILE_PATTERN)))
if not archivos:
    raise FileNotFoundError(
        f"No se encontraron archivos con el patrón '{FILE_PATTERN}' en '{STATIONS_FOLDER}'. "
        "Revisa la ruta y que los archivos existan."
    )

print(f"Se encontraron {len(archivos)} archivo(s):")
for a in archivos:
    print(' -', a, '->', extraer_nombre_estacion(a))

estaciones_data = {extraer_nombre_estacion(a): cargar_estacion(a) for a in archivos}


Se encontraron 15 archivo(s):
 - ../data_estaciones_e4\BD_CE.csv -> BD_CE
 - ../data_estaciones_e4\BD_NE.csv -> BD_NE
 - ../data_estaciones_e4\BD_NE2.csv -> NE2
 - ../data_estaciones_e4\BD_NE3.csv -> NE3
 - ../data_estaciones_e4\BD_NO.csv -> BD_NO
 - ../data_estaciones_e4\BD_NO2.csv -> NO2
 - ../data_estaciones_e4\BD_NO3.csv -> NO3
 - ../data_estaciones_e4\BD_NTE.csv -> BD_NTE
 - ../data_estaciones_e4\BD_NTE2.csv -> NTE2
 - ../data_estaciones_e4\BD_SE.csv -> BD_SE
 - ../data_estaciones_e4\BD_SE2.csv -> SE2
 - ../data_estaciones_e4\BD_SE3.csv -> SE3
 - ../data_estaciones_e4\BD_SO.csv -> BD_SO
 - ../data_estaciones_e4\BD_SO2.csv -> SO2
 - ../data_estaciones_e4\BD_SUR.csv -> BD_SUR


# 1.1 Limpieza

In [4]:
# Rangos validos segun SIMA. Los valores fuera de estos limites se reemplazan por NaN.
rangos_validos = {
    'PM10': (0, 999),
    'PM2.5': (0, 999),
    'O3': (0, 185),
    'NO': (0, 500),
    'NO2': (0, 200),
    'SO2': (0, 405),
    'CO': (0, 20),
    'RH': (0, 100),
    'WSR': (0, 75),
    'TOUT': (-6.5, 45.5),
    'SR': (0, 1.26),
    'WDR': (0, 360),
    'RAINF': (0, 80)
}

for nombre, estacion in estaciones_data.items():
    valores_eliminados = 0

    for columna, (minimo, maximo) in rangos_validos.items():
        if columna in estacion.columns:
            valores = pd.to_numeric(estacion[columna], errors='coerce')
            fuera_de_rango = (valores < minimo) | (valores > maximo)
            valores_eliminados += int(fuera_de_rango.sum())
            estacion[columna] = valores.mask(fuera_de_rango)

    estaciones_data[nombre] = estacion
    print(f'{nombre}: {valores_eliminados} valores fuera de rango reemplazados por NaN')

BD_CE: 10 valores fuera de rango reemplazados por NaN
BD_NE: 49 valores fuera de rango reemplazados por NaN
NE2: 74 valores fuera de rango reemplazados por NaN
NE3: 95 valores fuera de rango reemplazados por NaN
BD_NO: 275 valores fuera de rango reemplazados por NaN
NO2: 39 valores fuera de rango reemplazados por NaN
NO3: 160 valores fuera de rango reemplazados por NaN
BD_NTE: 1062 valores fuera de rango reemplazados por NaN
NTE2: 4 valores fuera de rango reemplazados por NaN
BD_SE: 29 valores fuera de rango reemplazados por NaN
SE2: 19 valores fuera de rango reemplazados por NaN
SE3: 4 valores fuera de rango reemplazados por NaN
BD_SO: 235 valores fuera de rango reemplazados por NaN
SO2: 589 valores fuera de rango reemplazados por NaN
BD_SUR: 10 valores fuera de rango reemplazados por NaN


## 2. Cobertura y calidad de datos por estación

In [5]:
def resumen_cobertura(df):
    variables_clave = ['PM10', 'PM2.5', 'RH', 'WSR', 'WDR']
    cobertura = {f'pct_no_nulo_{v}': round(100 * df[v].notna().mean(), 1) for v in variables_clave}
    return {
        'n_obs': len(df),
        'anio_inicio': int(df['anio'].min()),
        'anio_fin': int(df['anio'].max()),
        'n_anios': df['anio'].nunique(),
        **cobertura,
    }

tabla_cobertura = pd.DataFrame({
    est: resumen_cobertura(df) for est, df in estaciones_data.items()
}).T
tabla_cobertura.index.name = 'estacion'
tabla_cobertura


,n_obs,anio_inicio,anio_fin,n_anios,pct_no_nulo_PM10,pct_no_nulo_PM2.5,pct_no_nulo_RH,pct_no_nulo_WSR,pct_no_nulo_WDR
estacion,,,,,,,,,
BD_CE,52604.000,2020.000,2025.000,6.000,97.200,85.900,97.700,97.900,98.000
BD_NE,52594.000,2020.000,2025.000,6.000,96.400,92.800,94.500,86.600,92.200
NE2,52592.000,2020.000,2025.000,6.000,94.900,67.200,96.100,86.800,90.500
NE3,43816.000,2021.000,2025.000,5.000,94.300,5.000,96.200,93.500,96.200
BD_NO,52596.000,2020.000,2025.000,6.000,94.400,54.300,87.500,94.300,89.500
NO2,52593.000,2020.000,2025.000,6.000,97.300,85.400,95.500,96.100,92.900
NO3,27044.000,2022.000,2025.000,4.000,87.100,13.100,61.300,90.700,97.900
BD_NTE,52594.000,2020.000,2025.000,6.000,91.500,84.400,41.500,81.900,79.500
NTE2,52592.000,2020.000,2025.000,6.000,96.500,85.600,98.400,98.800,98.700


## 3. Régimen de humedad (RH)

Mediana de RH y porcentaje de tiempo por encima del umbral de saturación configurado. Estaciones que ya pasan mucho tiempo por encima del umbral tienen **menos margen** para que añadir humedad (vía vegetación) siga aportando beneficio sobre PM10.

In [6]:
def criterio_humedad(df):
    rh = df['RH'].dropna()
    if len(rh) == 0:
        return {'RH_mediana': np.nan, 'pct_tiempo_sobre_umbral_RH': np.nan}
    return {
        'RH_mediana': rh.median(),
        'pct_tiempo_sobre_umbral_RH': round(100 * (rh >= RH_SATURATION_THRESHOLD).mean(), 1),
    }

tabla_humedad = pd.DataFrame({
    est: criterio_humedad(df) for est, df in estaciones_data.items()
}).T
tabla_humedad.index.name = 'estacion'
tabla_humedad


,RH_mediana,pct_tiempo_sobre_umbral_RH
estacion,,
BD_CE,56.000,19.600
BD_NE,65.000,34.700
NE2,58.000,23.400
NE3,64.000,34.700
BD_NO,56.000,22.500
NO2,57.000,23.600
NO3,59.000,20.800
BD_NTE,50.000,15.200
NTE2,57.000,21.600


## 4. Régimen de ventilación (WSR)

El umbral de "estancamiento" se calcula como un percentil bajo de WSR sobre el conjunto de **todas** las estaciones cargadas (más robusto que asumir un valor absoluto sin conocer bien las unidades/calibración de cada sensor). Ajustar `WSR_STAGNATION_PERCENTILE` en la configuración si se prefiere otro criterio.

In [7]:
wsr_pool = pd.concat([df['WSR'] for df in estaciones_data.values()]).dropna()
umbral_estancamiento = np.percentile(wsr_pool, WSR_STAGNATION_PERCENTILE)
print(f'Umbral de estancamiento (percentil {WSR_STAGNATION_PERCENTILE} de WSR en toda la red): '
      f'{umbral_estancamiento:.2f}')

def criterio_ventilacion(df):
    wsr = df['WSR'].dropna()
    if len(wsr) == 0:
        return {'WSR_mediana': np.nan, 'pct_tiempo_estancado': np.nan}
    return {
        'WSR_mediana': wsr.median(),
        'pct_tiempo_estancado': round(100 * (wsr <= umbral_estancamiento).mean(), 1),
    }

tabla_ventilacion = pd.DataFrame({
    est: criterio_ventilacion(df) for est, df in estaciones_data.items()
}).T
tabla_ventilacion.index.name = 'estacion'
tabla_ventilacion


Umbral de estancamiento (percentil 10 de WSR en toda la red): 2.30


,WSR_mediana,pct_tiempo_estancado
estacion,,
BD_CE,6.900,8.600
BD_NE,8.200,5.500
NE2,8.000,8.100
NE3,7.200,18.500
BD_NO,7.200,9.800
NO2,9.300,6.800
NO3,9.600,0.800
BD_NTE,7.000,8.800
NTE2,7.400,3.000


## 5. Fracción gruesa de PM10

Se calcula solo con observaciones donde PM10 y PM2.5 están disponibles simultáneamente y PM10 ≥ PM2.5 (filtro de consistencia física). Una fracción gruesa alta sugiere que domina partícula gruesa (polvo/erosión), más sensible al mecanismo de deposición asistida por humedad que la partícula fina de fuentes de combustión.

In [8]:
def criterio_fraccion_gruesa(df):
    sub = df[['PM10', 'PM2.5']].dropna()
    sub = sub[sub['PM10'] >= sub['PM2.5']]
    if len(sub) < 30:
        return {'n_pares_validos': len(sub), 'fraccion_gruesa_mediana': np.nan}
    fraccion = (sub['PM10'] - sub['PM2.5']) / sub['PM10']
    return {
        'n_pares_validos': len(sub),
        'fraccion_gruesa_mediana': round(fraccion.median(), 3),
    }

tabla_fraccion = pd.DataFrame({
    est: criterio_fraccion_gruesa(df) for est, df in estaciones_data.items()
}).T
tabla_fraccion.index.name = 'estacion'
tabla_fraccion


,n_pares_validos,fraccion_gruesa_mediana
estacion,,
BD_CE,44673.000,0.621
BD_NE,48366.000,0.635
NE2,34431.000,0.685
NE3,2165.000,0.583
BD_NO,28389.000,0.611
NO2,44483.000,0.687
NO3,3520.000,0.839
BD_NTE,43406.000,0.699
NTE2,44041.000,0.705


## 6. Concentración por dirección de viento

Se agrupa `WDR` (grados, 0–360) en 8 sectores de compás y se calcula el PM10 promedio por sector. Se reporta el sector dominante y un índice de concentración (razón entre el PM10 promedio del sector más contaminado y el promedio general).

In [9]:
SECTORES = ['N', 'NE', 'E', 'SE', 'S', 'SO', 'O', 'NO']

def asignar_sector(grados):
    if pd.isna(grados):
        return np.nan
    idx = int(((grados + 22.5) % 360) // 45)
    return SECTORES[idx]

def criterio_direccion_viento(df):
    sub = df[['WDR', 'PM10']].dropna()
    if len(sub) < 30:
        return {'sector_dominante': np.nan, 'indice_concentracion_direccional': np.nan}
    sub = sub.copy()
    sub['sector'] = sub['WDR'].apply(asignar_sector)
    pm10_por_sector = sub.groupby('sector')['PM10'].mean()
    pm10_promedio_general = sub['PM10'].mean()
    sector_dominante = pm10_por_sector.idxmax()
    indice = pm10_por_sector.max() / pm10_promedio_general
    return {
        'sector_dominante': sector_dominante,
        'indice_concentracion_direccional': round(indice, 2),
    }

tabla_direccion = pd.DataFrame({
    est: criterio_direccion_viento(df) for est, df in estaciones_data.items()
}).T
tabla_direccion.index.name = 'estacion'
tabla_direccion


,sector_dominante,indice_concentracion_direccional
estacion,,
BD_CE,O,1.200
BD_NE,O,1.660
NE2,NO,1.060
NE3,O,1.350
BD_NO,NO,1.440
NO2,SO,1.190
NO3,SO,1.270
BD_NTE,NO,1.420
NTE2,SO,1.460


## 7. Matriz de priorización final

Se normaliza cada criterio a una escala 0–1 (1 = más favorable para que aumentar vegetación ayude a reducir PM10, según la literatura) y se combina en un score ponderado con los pesos definidos en la configuración. Las columnas `NDVI_actual` y `EVI_actual` quedan vacías para llenarse manualmente.

In [ ]:
def normalizar_favorable_bajo(serie):
    """Para criterios donde un valor MÁS BAJO es más favorable (ej. % tiempo sobre umbral RH,
    % tiempo estancado, índice de concentración direccional): invierte y normaliza a 0-1."""
    s = serie.astype(float)
    rango = s.max() - s.min()
    if rango == 0 or pd.isna(rango):
        return pd.Series(0.5, index=s.index)
    return 1 - (s - s.min()) / rango

def normalizar_favorable_alto(serie):
    """Para criterios donde un valor MÁS ALTO es más favorable (ej. fracción gruesa de PM10)."""
    s = serie.astype(float)
    rango = s.max() - s.min()
    if rango == 0 or pd.isna(rango):
        return pd.Series(0.5, index=s.index)
    return (s - s.min()) / rango

matriz = tabla_cobertura.join([tabla_humedad, tabla_ventilacion, tabla_fraccion, tabla_direccion])

matriz['score_humedad'] = normalizar_favorable_bajo(matriz['pct_tiempo_sobre_umbral_RH'])
matriz['score_ventilacion'] = normalizar_favorable_bajo(matriz['pct_tiempo_estancado'])
matriz['score_fraccion_gruesa'] = normalizar_favorable_alto(matriz['fraccion_gruesa_mediana'])
matriz['score_fuente_direccional'] = normalizar_favorable_bajo(matriz['indice_concentracion_direccional'])

matriz['score_compuesto'] = (
    PESOS['humedad'] * matriz['score_humedad']
    + PESOS['ventilacion'] * matriz['score_ventilacion']
    + PESOS['fraccion_gruesa'] * matriz['score_fraccion_gruesa']
    + PESOS['fuente_direccional'] * matriz['score_fuente_direccional']
)

matriz['NDVI_actual'] = np.nan
matriz['EVI_actual'] = np.nan

def clasificar_prioridad(score):
    if pd.isna(score):
        return 'sin datos suficientes'
    if score >= 0.66:
        return 'alta'
    elif score >= 0.33:
        return 'media'
    return 'baja'

matriz['prioridad'] = matriz['score_compuesto'].apply(clasificar_prioridad)

columnas_finales = [
    'n_obs', 'anio_inicio', 'anio_fin', 'n_anios',
    'RH_mediana', 'pct_tiempo_sobre_umbral_RH',
    'WSR_mediana', 'pct_tiempo_estancado',
    'fraccion_gruesa_mediana', 'sector_dominante', 'indice_concentracion_direccional',
    'NDVI_actual', 'EVI_actual',
    'score_compuesto', 'prioridad',
]
matriz_final = matriz[columnas_finales].sort_values('score_compuesto', ascending=False)
matriz_final


,n_obs,anio_inicio,anio_fin,n_anios,RH_mediana,pct_tiempo_sobre_umbral_RH,WSR_mediana,pct_tiempo_estancado,fraccion_gruesa_mediana,sector_dominante,indice_concentracion_direccional,NDVI_actual,EVI_actual,score_compuesto,prioridad
estacion,,,,,,,,,,,,,,,
NO3,27044.000,2022.000,2025.000,4.000,59.000,20.800,9.600,0.800,0.839,SO,1.270,NaN,NaN,0.841,alta
SO2,52592.000,2020.000,2025.000,6.000,55.000,16.700,9.500,9.000,0.700,O,1.110,NaN,NaN,0.750,alta
NE2,52592.000,2020.000,2025.000,6.000,58.000,23.400,8.000,8.100,0.685,NO,1.060,NaN,NaN,0.678,alta
BD_NTE,52594.000,2020.000,2025.000,6.000,50.000,15.200,7.000,8.800,0.699,NO,1.420,NaN,NaN,0.641,media
NO2,52593.000,2020.000,2025.000,6.000,57.000,23.600,9.300,6.800,0.687,SO,1.190,NaN,NaN,0.635,media
BD_CE,52604.000,2020.000,2025.000,6.000,56.000,19.600,6.900,8.600,0.621,O,1.200,NaN,NaN,0.601,media
NTE2,52592.000,2020.000,2025.000,6.000,57.000,21.600,7.400,3.000,0.705,SO,1.460,NaN,NaN,0.600,media
BD_SE,52598.000,2020.000,2025.000,6.000,62.000,29.400,9.500,2.000,0.636,NO,1.100,NaN,NaN,0.592,media
BD_SUR,52591.000,2020.000,2025.000,6.000,59.000,23.700,5.500,17.000,0.674,N,1.230,NaN,NaN,0.512,media


## 8. Exportar la matriz

In [11]:
OUTPUT_PATH = 'matriz_priorizacion_estaciones.csv'
matriz_final.to_csv(OUTPUT_PATH)
print(f'Matriz guardada en: {OUTPUT_PATH}')


Matriz guardada en: matriz_priorizacion_estaciones.csv
